### Extract Data From the Orders JSON File
1. Query Orders File using JSON Format
2. Query Orders File using TEXT Format
3. Create Orders View in Bronze Schema
4. Parse corrupted JSON file using PySpark

###### 1. Query Orders File using JSON Format

In [0]:
SELECT * FROM json.`/Volumes/gizmobox/landing/operational_data/orders/`

##### 2. Query Orders File using TEXT Format

In [0]:
SELECT * FROM text.`/Volumes/gizmobox/landing/operational_data/orders/`

##### 3. Create Orders View in Bronze Schema

In [0]:
CREATE OR REPLACE VIEW gizmobox.bronze.v_orders
AS 
SELECT * FROM text.`/Volumes/gizmobox/landing/operational_data/orders/`

In [0]:
SELECT * FROM gizmobox.bronze.v_orders;

4. Parse corrupted JSON file using PySpark

In [0]:
%python


# Load with permissive mode and multiline enabled for safety
raw_df = spark.read.option("mode", "PERMISSIVE").json("abfss://gizmobox@deawrkspcextdl.dfs.core.windows.net/landing/operational_data/orders/orders_2024_10.json")
display(raw_df)

In [0]:
%python
raw_df.printSchema()

In [0]:
%python
from pyspark.sql.types import *

# Define schema treating problematic fields as strings
custom_schema = StructType([
    StructField("order_id", LongType(), True),
    StructField("customer_id", LongType(), True),
    StructField("order_date", StringType(), True),  # Keep as String
    StructField("transaction_timestamp", StringType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("payment_method", StringType(), True),
    StructField("items", ArrayType(
        StructType([
            StructField("item_id", LongType(), True),
            StructField("name", StringType(), True),
            StructField("category", StringType(), True),
            StructField("price", DoubleType(), True),
            StructField("quantity", LongType(), True),
            StructField("details", MapType(StringType(), StringType()), True)
        ])
    )),
    StructField("order_status", StringType(), True)
])

df = spark.read.schema(custom_schema).json("abfss://gizmobox@deawrkspcextdl.dfs.core.windows.net/landing/operational_data/orders/orders_2024_10.json")
display(df)

In [0]:
%python
from pyspark.sql.functions import *

df = df.withColumn("order_date", to_date(col("order_date"), "yyyy-MM-dd"))
display(df)

In [0]:
%python
flat_df = df.select("order_id", "customer_id", "order_date", "transaction_timestamp", 
                    "total_amount", "payment_method", "order_status", explode("items").alias("item"))
display(flat_df)

In [0]:
%python

flat_df2 = flat_df.select(
    "order_id", "customer_id", "order_date", "transaction_timestamp", "total_amount", "payment_method", "order_status",
    "item.item_id", "item.name", "item.category", "item.price", "item.quantity", "item.details.brand", "item.details.color"
)

display(flat_df2)